<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/VoxCPM2_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# VoxCPM2 — Tokenizer-Free Multilingual TTS with Voice Design & Cloning

VoxCPM2 is a **2B-parameter tokenizer-free TTS** model from [OpenBMB](https://github.com/OpenBMB) (Tsinghua / ModelBest inc), built on a [MiniCPM-4](https://huggingface.co/openbmb/MiniCPM4-0.5B) backbone. It models speech in a **continuous latent space** (no discrete tokens), enabling four flagship capabilities: **plain TTS** with content-aware prosody, **Voice Design** from a text description alone, **Controllable Cloning** from a short reference clip, and **Ultimate Cloning** that preserves every vocal nuance. Native support for **30 languages** and **48 kHz studio-quality output**. Apache-2.0 — free for commercial use.

> "VoxCPM achieves competitive or state-of-the-art results on public zero-shot TTS benchmarks." — [arXiv 2509.24650](https://arxiv.org/abs/2509.24650) (VoxCPM technical report)

**Upstream**: [OpenBMB/VoxCPM](https://github.com/OpenBMB/VoxCPM) · [openbmb/VoxCPM2](https://huggingface.co/openbmb/VoxCPM2) · [arXiv 2509.24650](https://arxiv.org/abs/2509.24650) · [Apache-2.0](https://github.com/OpenBMB/VoxCPM/blob/main/LICENSE)

## What this notebook does

- **Four generation modes** in one notebook:
  - **TTS** — plain text → speech, voice inferred from content
  - **Voice Design** — type a description in parens → model creates a matching voice from scratch (no ref audio needed)
  - **Voice Clone** — short ref clip → clone the timbre
  - **Ultimate Clone** — ref clip + transcript + prompt audio → preserve every vocal nuance
- **Two model versions** in one notebook via dropdown:
  - **VoxCPM2** (default, 2B, 30 langs, 48 kHz, ~8 GB VRAM)
  - **VoxCPM-0.5B** (legacy, 0.5B, ZH/EN, 16 kHz, ~5 GB VRAM)
- **Streaming** output for long texts (RTF as low as 0.13 on RTX 4090 with Nano-vLLM)
- **8 tabs**: TTS / Voice Design / Voice Clone / Ultimate Clone / Streaming / Batch / VRAM / Help
- **Optional text normalization** (WeTextProcessing) for numbers/abbreviations
- **Optional denoiser** (ZipEnhancer) for noisy reference audio

## What this notebook does NOT do

- **LoRA fine-tuning UI** (the upstream has one, but we skip it to stay focused on inference)
- **Production-grade streaming server** (use [vLLM-Omni](https://github.com/vllm-project/vllm-omni) for that)
- **Multimodal / dialog** (this is TTS-only)

## How to use

1. Run **STEP 1** (installs `voxcpm`, `modelscope`, `soundfile`).
2. *Optional* — run **STEP 2** to pre-cache weights to Drive (~5 GB for VoxCPM2).
3. Run **STEP 3** to load the core functions.
4. Run **STEP 4** for the interactive Gradio UI.
5. *Optional* — run **STEP 5** to prevent Colab from disconnecting.
6. *Optional* — run **STEP 6** for a one-off generation, or **STEP 7** for batch.

> **Tip:** First call downloads the ZipEnhancer denoiser (~50 MB) from ModelScope on first use. Set `denoise=False` in the UI to skip it.

## Citation

```bibtex
@article{voxcpm2025,
  title  = {VoxCPM: Tokenizer-Free TTS for Context-Aware Speech Generation and True-to-Life Voice Cloning},
  author = {{VoxCPM Team}},
  journal= {arXiv preprint arXiv:2509.24650},
  year   = {2025}
}
```


In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Installs the `voxcpm` package (with `modelscope` for the optional ZipEnhancer denoiser), `soundfile`, and the `tqdm` progress bar. ~1-2 min first time.

import os, sys, subprocess, time

# GPU auto-detect
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[GPU] torch {torch.__version__} · device = {DEVICE}', flush=True)
if DEVICE == 'cuda':
    try:
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'[GPU] {name} ({vram:.1f} GB VRAM)', flush=True)
    except Exception as e:
        print(f'[GPU] (could not read device name: {e})', flush=True)

# Drive cache setup
DRIVE = '/content/drive/MyDrive/AEI_TTS_Cache/huggingface'
os.environ.setdefault('HF_HOME', DRIVE)
os.makedirs(DRIVE, exist_ok=True)
print(f'[Cache] HF_HOME = {DRIVE}', flush=True)

# Output dirs
os.makedirs('/content/audio_out', exist_ok=True)
os.makedirs('/content/ref_audio', exist_ok=True)
print('[Setup] /content/audio_out and /content/ref_audio ready.', flush=True)

# Install voxcpm + deps
# VoxCPM requires Python 3.10+ and torch 2.5+. We pin tqdm + soundfile.
print('\n[pip] Installing voxcpm + modelscope + soundfile...', flush=True)
t0 = time.time()
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'voxcpm>=2.0.0', 'modelscope>=1.22.0', 'soundfile',
     'numpy', 'tqdm', 'hf_transfer', 'gradio>=4.0',
     'inflect'],
    check=True,
)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print(f'[pip] Done in {time.time()-t0:.1f}s', flush=True)

# Sanity-check the imports
try:
    import voxcpm
    print(f'[voxcpm] version = {voxcpm.__version__ if hasattr(voxcpm, "__version__") else "(unknown)"}', flush=True)
except Exception as e:
    print(f'[voxcpm] FAILED: {e}', flush=True)
    raise

# Check Python version
py_ver = sys.version_info
print(f'[Python] {py_ver.major}.{py_ver.minor}.{py_ver.micro}', flush=True)
if py_ver < (3, 10) or py_ver >= (3, 13):
    print('[Python] WARNING: VoxCPM requires Python 3.10-3.12', flush=True)

print('\n[Done] STEP 1 complete. Move on to STEP 2 (pre-cache) or STEP 3 (core).', flush=True)


In [ ]:
# @title STEP 2 — Pre-cache Model Weights
# @markdown Downloads the VoxCPM2 model + ZipEnhancer denoiser to the local cache (or Drive if mounted). Resumable.

# @markdown **VoxCPM2 size: ~5 GB · VoxCPM-0.5B size: ~2 GB · ZipEnhancer: ~50 MB (optional).**

import os
from huggingface_hub import snapshot_download

# VoxCPM model repos
MODELS = {
    'openbmb/VoxCPM2':     'VoxCPM2 (2B, 30 langs)',
    'openbmb/VoxCPM-0.5B': 'VoxCPM-0.5B (legacy)',
}

# Pre-cache the main model (VoxCPM2) — others can be lazy-downloaded
for repo_id, label in MODELS.items():
    print(f'\n[Download] {label} ({repo_id})...', flush=True)
    try:
        path = snapshot_download(repo_id)
        print(f'  ✓ cached at {path}', flush=True)
    except Exception as e:
        print(f'  ✗ {e} (will lazy-download on first use)', flush=True)

# Pre-cache the ZipEnhancer denoiser (from ModelScope)
print('\n[Download] ZipEnhancer denoiser (ModelScope)...', flush=True)
try:
    from modelscope import snapshot_download as ms_snapshot_download
    denoiser_path = ms_snapshot_download('iic/speech_zipenhancer_ans_multiloss_16k_base')
    print(f'  ✓ cached at {denoiser_path}', flush=True)
except Exception as e:
    print(f'  ✗ {e} (skip — denoise will be disabled)', flush=True)

print('\n[Done] STEP 2 complete. All weights cached.', flush=True)


In [ ]:
# @title STEP 3 — Core Functions (lazy model loading)
# @markdown Defines the inference wrappers. The VoxCPM model is loaded on first use, not at import time. This cell is fast.

import os, time, gc
import numpy as np
import torch
import soundfile as sf
from typing import Optional, Tuple, Generator

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── VoxCPM versions table ────────────────────────────────────────
# (id, hf_model_id, display, params, sample_rate, langs, vram_gb, features)
VERSIONS = [('voxcpm2', 'openbmb/VoxCPM2', 'VoxCPM2 (latest, 2B, 30 langs, 48 kHz)', '2B', 48000, 30, 8, 'Voice Design + Controllable Cloning + Ultimate Cloning'), ('voxcpm-0.5b', 'openbmb/VoxCPM-0.5B', 'VoxCPM-0.5B (legacy, 0.5B, ZH/EN, 16 kHz)', '0.5B', 16000, 2, 5, 'Continuation Cloning only')]

# ── Voice design presets (use in TTS textbox, e.g. "(A young woman)Hello!") ─
VOICE_DESIGN_PRESETS = ['(A young woman, gentle and sweet voice)', '(An elderly British man, deep and slow)', '(A cheerful teenage girl, energetic and bright)', '(A professional male news anchor, neutral and clear)', '(A warm grandmother, soft and caring)', '(A confident young man, energetic and persuasive)', '(A mysterious whisper, low and breathy)', '(A child, high-pitched and excited)']

# ── Lazy model loader ───────────────────────────────────────────
MODEL = None  # VoxCPM instance
MODEL_VERSION = None  # str id


def get_voxcpm(version_id: str = 'voxcpm2', load_denoiser: bool = True):
    """Lazily load a VoxCPM model. Idempotent — only reloads if version changes.
    version_id: 'voxcpm2' | 'voxcpm-0.5b'
    load_denoiser: if True, also loads the ZipEnhancer acoustic denoiser (~50 MB)
    """
    global MODEL, MODEL_VERSION
    if MODEL is not None and MODEL_VERSION == version_id:
        return MODEL

    # Free previous model if switching versions
    if MODEL is not None:
        free_voxcpm()

    spec = next((v for v in VERSIONS if v[0] == version_id), None)
    if spec is None:
        raise ValueError(f"Unknown version_id {version_id!r}. Choose from {[v[0] for v in VERSIONS]}")

    repo_id = spec[1]
    from voxcpm import VoxCPM
    print(f'[VoxCPM] Loading {spec[2]} from {repo_id} (denoiser={load_denoiser})...', flush=True)
    t0 = time.time()
    MODEL = VoxCPM.from_pretrained(
        hf_model_id=repo_id,
        load_denoiser=load_denoiser,
        optimize=False,  # torch.compile is slow on first call; skip for notebook speed
        device=DEVICE,
    )
    MODEL_VERSION = version_id
    print(f'[VoxCPM] {version_id} ready in {time.time()-t0:.1f}s', flush=True)
    return MODEL


def free_voxcpm():
    """Release the loaded VoxCPM model from VRAM."""
    global MODEL, MODEL_VERSION
    if MODEL is not None:
        del MODEL
        MODEL = None
        MODEL_VERSION = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print('[Free] VoxCPM model released', flush=True)


# ── Save helper ──────────────────────────────────────────────────
def save_wav(audio: np.ndarray, sr: int, name: str) -> str:
    """Save a 1-D float numpy array as a 16-bit PCM WAV."""
    path = os.path.join('/content/audio_out', name)
    sf.write(path, audio, sr, subtype='PCM_16')
    return path


# ── 4 generation modes ──────────────────────────────────────────
def synthesize_tts(text: str,
                   cfg_value: float = 2.0,
                   inference_timesteps: int = 10,
                   normalize: bool = False,
                   denoise: bool = False,
                   version_id: str = 'voxcpm2',
                   progress=None) -> Tuple[np.ndarray, int, dict]:
    """Plain TTS — model infers the voice from the text itself.
    No reference audio. Best for general-purpose narration."""
    if not text or not text.strip():
        raise ValueError('Text is empty.')
    if progress:
        progress(0.0, 'Loading model...')
    model = get_voxcpm(version_id)
    if progress:
        progress(0.2, 'Generating...')
    t0 = time.time()
    audio = model.generate(
        text=text.strip(),
        cfg_value=float(cfg_value),
        inference_timesteps=int(inference_timesteps),
        normalize=bool(normalize),
        denoise=bool(denoise),
    )
    elapsed = time.time() - t0
    if progress:
        progress(1.0, f'Done in {elapsed:.1f}s')
    sr = model.tts_model.sample_rate
    return np.asarray(audio, dtype=np.float32), sr, {
        'mode': 'TTS',
        'elapsed': elapsed,
        'sample_rate': sr,
        'cfg': cfg_value,
        'timesteps': inference_timesteps,
    }


def synthesize_voice_design(description: str,
                             text: str,
                             cfg_value: float = 2.0,
                             inference_timesteps: int = 10,
                             normalize: bool = False,
                             denoise: bool = False,
                             version_id: str = 'voxcpm2',
                             progress=None) -> Tuple[np.ndarray, int, dict]:
    """Voice Design — create a voice from a natural-language description.
    Put the description in parens at the start of the text: '(...)The text to speak.'
    VoxCPM-0.5B does NOT support this mode."""
    if not text or not text.strip():
        raise ValueError('Text is empty.')
    if not description or not description.strip():
        raise ValueError('Description is empty.')
    if version_id == 'voxcpm-0.5b':
        raise RuntimeError('Voice Design is only supported by VoxCPM2. Switch the model version in the UI.')
    if progress:
        progress(0.0, 'Loading model...')
    model = get_voxcpm(version_id)
    # Strip any surrounding parens the user may have included, then re-wrap.
    # (A young woman) → A young woman → (A young woman)
    inner = description.strip()
    if inner.startswith('(') and inner.endswith(')'):
        inner = inner[1:-1].strip()
    full_text = f'({inner}){text.strip()}'
    if progress:
        progress(0.2, 'Generating with voice design...')
    t0 = time.time()
    audio = model.generate(
        text=full_text,
        cfg_value=float(cfg_value),
        inference_timesteps=int(inference_timesteps),
        normalize=bool(normalize),
        denoise=bool(denoise),
    )
    elapsed = time.time() - t0
    if progress:
        progress(1.0, f'Done in {elapsed:.1f}s')
    sr = model.tts_model.sample_rate
    return np.asarray(audio, dtype=np.float32), sr, {
        'mode': 'Voice Design',
        'description': description,
        'elapsed': elapsed,
        'sample_rate': sr,
    }


def synthesize_voice_clone(text: str,
                           reference_audio_path: str,
                           cfg_value: float = 2.0,
                           inference_timesteps: int = 10,
                           normalize: bool = False,
                           denoise: bool = False,
                           version_id: str = 'voxcpm2',
                           progress=None) -> Tuple[np.ndarray, int, dict]:
    """Voice Clone — clone the timbre from a short reference clip.
    VoxCPM-0.5B does NOT support reference_wav_path."""
    if not text or not text.strip():
        raise ValueError('Text is empty.')
    if not reference_audio_path or not os.path.exists(reference_audio_path):
        raise ValueError('Reference audio path is required and must exist.')
    if version_id == 'voxcpm-0.5b':
        raise RuntimeError('reference_wav_path is only supported by VoxCPM2.')
    if progress:
        progress(0.0, 'Loading model...')
    model = get_voxcpm(version_id)
    if progress:
        progress(0.2, 'Cloning voice...')
    t0 = time.time()
    audio = model.generate(
        text=text.strip(),
        reference_wav_path=reference_audio_path,
        cfg_value=float(cfg_value),
        inference_timesteps=int(inference_timesteps),
        normalize=bool(normalize),
        denoise=bool(denoise),
    )
    elapsed = time.time() - t0
    if progress:
        progress(1.0, f'Done in {elapsed:.1f}s')
    sr = model.tts_model.sample_rate
    return np.asarray(audio, dtype=np.float32), sr, {
        'mode': 'Voice Clone',
        'reference': os.path.basename(reference_audio_path),
        'elapsed': elapsed,
        'sample_rate': sr,
    }


def synthesize_ultimate_clone(text: str,
                             reference_audio_path: str,
                             prompt_audio_path: str,
                             prompt_text: str,
                             cfg_value: float = 2.0,
                             inference_timesteps: int = 10,
                             normalize: bool = False,
                             denoise: bool = False,
                             version_id: str = 'voxcpm2',
                             progress=None) -> Tuple[np.ndarray, int, dict]:
    """Ultimate Clone — provide reference audio + transcript + prompt audio.
    Preserves every vocal nuance of the reference (continuation-based)."""
    if not text or not text.strip():
        raise ValueError('Text is empty.')
    if not reference_audio_path or not os.path.exists(reference_audio_path):
        raise ValueError('Reference audio is required.')
    if not prompt_audio_path or not os.path.exists(prompt_audio_path):
        raise ValueError('Prompt audio is required for Ultimate Clone.')
    if not prompt_text or not prompt_text.strip():
        raise ValueError('Prompt text (transcript of the reference) is required.')
    if version_id == 'voxcpm-0.5b':
        raise RuntimeError('Ultimate Clone is only supported by VoxCPM2.')
    if progress:
        progress(0.0, 'Loading model...')
    model = get_voxcpm(version_id)
    if progress:
        progress(0.2, 'Ultimate cloning (prompt + reference)...')
    t0 = time.time()
    audio = model.generate(
        text=text.strip(),
        prompt_wav_path=prompt_audio_path,
        prompt_text=prompt_text.strip(),
        reference_wav_path=reference_audio_path,
        cfg_value=float(cfg_value),
        inference_timesteps=int(inference_timesteps),
        normalize=bool(normalize),
        denoise=bool(denoise),
    )
    elapsed = time.time() - t0
    if progress:
        progress(1.0, f'Done in {elapsed:.1f}s')
    sr = model.tts_model.sample_rate
    return np.asarray(audio, dtype=np.float32), sr, {
        'mode': 'Ultimate Clone',
        'reference': os.path.basename(reference_audio_path),
        'prompt': os.path.basename(prompt_audio_path),
        'elapsed': elapsed,
        'sample_rate': sr,
    }


def synthesize_streaming(text: str,
                        version_id: str = 'voxcpm2',
                        cfg_value: float = 2.0,
                        inference_timesteps: int = 10,
                        normalize: bool = False) -> Generator[Tuple[np.ndarray, int], None, None]:
    """Streaming — yields audio chunks as they're generated."""
    if not text or not text.strip():
        return
    model = get_voxcpm(version_id)
    sr = model.tts_model.sample_rate
    for chunk in model.generate_streaming(
        text=text.strip(),
        cfg_value=float(cfg_value),
        inference_timesteps=int(inference_timesteps),
        normalize=bool(normalize),
    ):
        yield np.asarray(chunk, dtype=np.float32), sr


def _materialize_audio(audio_obj):
    """Gradio can pass a filepath string, a (sr, ndarray) tuple, or just ndarray.
    Returns (sr, ndarray) or None. For a raw ndarray (no sr), returns
    (None, ndarray) so the caller can use the model's known sample rate."""
    if audio_obj is None:
        return None
    if isinstance(audio_obj, str):
        data, sr = sf.read(audio_obj)
        return int(sr), np.asarray(data, dtype=np.float32)
    if isinstance(audio_obj, (list, tuple)) and len(audio_obj) == 2:
        if isinstance(audio_obj[1], np.ndarray):
            return int(audio_obj[0]), audio_obj[1]
        return int(audio_obj[1]), np.asarray(audio_obj[0])
    if isinstance(audio_obj, np.ndarray):
        return None, audio_obj
    return None


print('[Core] Ready. Use STEP 4 for the UI, STEP 6 for quick test, STEP 7 for batch.')
print(f'[Core] {len(VERSIONS)} model versions, {len(VOICE_DESIGN_PRESETS)} voice design presets.')


In [ ]:
# @title STEP 4 — Gradio UI
# @markdown Interactive web app with eight tabs: **TTS** (plain text), **Voice Design** (description in parens), **Voice Clone** (ref audio), **Ultimate Clone** (ref + prompt + transcript), **Streaming** (long text streamed), **Batch** (zip of generations), **VRAM** (free model), **Help** (modes, languages, citation).

import os, sys, time, json, io, zipfile
import numpy as np
import soundfile as sf
import gradio as gr

# ── Colab / nest_asyncio so Gradio can run inside the notebook loop ──
try:
    import nest_asyncio
    nest_asyncio.apply()
except Exception:
    pass
try:
    from IPython.display import clear_output as _clear
    _clear()
except Exception:
    pass


# ── Tab 1: Plain TTS ─────────────────────────────────────────────
def ui_tts(text, version, cfg, timesteps, normalize, denoise, progress=gr.Progress()):
    if not text or not text.strip():
        raise gr.Error('Type some text first.')
    try:
        audio, sr, meta = synthesize_tts(text, cfg, timesteps, normalize, denoise, version, progress)
    except Exception as e:
        raise gr.Error(f'Generation failed: {e}')
    fname = f'tts_{int(time.time())}.wav'
    path = save_wav(audio, sr, fname)
    info = (f"mode = TTS · version = {version} · cfg = {cfg:.2f} · timesteps = {timesteps}\n"
            f"duration = {len(audio)/sr:.2f}s · sample_rate = {sr} Hz · {meta['elapsed']:.1f}s wall\n")
    return (sr, audio), path, info


# ── Tab 2: Voice Design ──────────────────────────────────────────
NONE_SENTINEL = '(none)'

def ui_voice_design(description, text, preset, version, cfg, timesteps, normalize, denoise,
                    progress=gr.Progress()):
    if not text or not text.strip():
        raise gr.Error('Type some text first.')
    # Custom description takes priority if non-empty; otherwise use the preset (unless '(none)').
    custom = (description or '').strip()
    if custom:
        desc = custom
    else:
        preset_clean = (preset or '').strip()
        desc = '' if preset_clean == NONE_SENTINEL else preset_clean
    if not desc:
        raise gr.Error('Provide a voice description (or pick a preset).')
    if version == 'voxcpm-0.5b':
        raise gr.Error('Voice Design is only supported by VoxCPM2. Switch model version.')
    try:
        audio, sr, meta = synthesize_voice_design(desc, text, cfg, timesteps, normalize, denoise, version, progress)
    except Exception as e:
        raise gr.Error(f'Generation failed: {e}')
    fname = f'design_{int(time.time())}.wav'
    path = save_wav(audio, sr, fname)
    info = (f"mode = Voice Design · version = {version}\n"
            f"description = {desc[:80]}{'…' if len(desc) > 80 else ''}\n"
            f"duration = {len(audio)/sr:.2f}s · sample_rate = {sr} Hz · {meta['elapsed']:.1f}s wall\n")
    return (sr, audio), path, info


# ── Tab 3: Voice Clone ───────────────────────────────────────────
def ui_voice_clone(text, ref_audio, version, cfg, timesteps, normalize, denoise,
                   progress=gr.Progress()):
    if not text or not text.strip():
        raise gr.Error('Type some text first.')
    if ref_audio is None:
        raise gr.Error('Upload a reference audio (5-30s of clear speech).')
    if version == 'voxcpm-0.5b':
        raise gr.Error('Voice Clone (reference_wav_path) is only supported by VoxCPM2. '
                       'VoxCPM-0.5B uses prompt_wav_path+prompt_text instead.')
    mat = _materialize_audio(ref_audio)
    if mat is None:
        raise gr.Error('Invalid reference audio.')
    ref_sr, ref_data = mat
    # If Gradio gave us a raw ndarray with no sr, fall back to the model's rate
    # (VoxCPM2 = 48 kHz, VoxCPM-0.5B = 16 kHz, but Voice Clone is V2-only here).
    if ref_sr is None:
        ref_sr = 48000
    ref_path = f'/content/ref_audio/clone_ref_{int(time.time()*1000)}.wav'
    sf.write(ref_path, ref_data, ref_sr, subtype='PCM_16')
    try:
        audio, sr, meta = synthesize_voice_clone(text, ref_path, cfg, timesteps, normalize, denoise, version, progress)
    except Exception as e:
        raise gr.Error(f'Generation failed: {e}')
    finally:
        try: os.remove(ref_path)
        except Exception: pass
    fname = f'clone_{int(time.time())}.wav'
    path = save_wav(audio, sr, fname)
    info = (f"mode = Voice Clone · version = {version} · ref = {meta['reference']}\n"
            f"duration = {len(audio)/sr:.2f}s · sample_rate = {sr} Hz · {meta['elapsed']:.1f}s wall\n")
    return (sr, audio), path, info


# ── Tab 4: Ultimate Clone ────────────────────────────────────────
def ui_ultimate_clone(text, ref_audio, prompt_audio, prompt_text, version, cfg, timesteps,
                      normalize, denoise, progress=gr.Progress()):
    if not text or not text.strip():
        raise gr.Error('Type some text first.')
    if ref_audio is None or prompt_audio is None:
        raise gr.Error('Upload both reference AND prompt audio.')
    if not prompt_text or not prompt_text.strip():
        raise gr.Error('Provide the transcript of the reference audio.')
    if version == 'voxcpm-0.5b':
        raise gr.Error('Ultimate Clone is only supported by VoxCPM2.')
    mat_r = _materialize_audio(ref_audio)
    mat_p = _materialize_audio(prompt_audio)
    if mat_r is None or mat_p is None:
        raise gr.Error('Invalid audio input.')
    ref_sr, ref_data = mat_r
    p_sr, p_data = mat_p
    # Fall back to V2's 48 kHz if no sr was attached (Ultimate Clone is V2-only)
    if ref_sr is None:
        ref_sr = 48000
    if p_sr is None:
        p_sr = 48000
    ref_path = f'/content/ref_audio/ulti_ref_{int(time.time()*1000)}.wav'
    p_path = f'/content/ref_audio/ulti_p_{int(time.time()*1000)}.wav'
    sf.write(ref_path, ref_data, ref_sr, subtype='PCM_16')
    sf.write(p_path, p_data, p_sr, subtype='PCM_16')
    try:
        audio, sr, meta = synthesize_ultimate_clone(text, ref_path, p_path, prompt_text,
                                                     cfg, timesteps, normalize, denoise, version, progress)
    except Exception as e:
        raise gr.Error(f'Generation failed: {e}')
    finally:
        for p in (ref_path, p_path):
            try: os.remove(p)
            except Exception: pass
    fname = f'ultimate_{int(time.time())}.wav'
    path = save_wav(audio, sr, fname)
    info = (f"mode = Ultimate Clone · ref = {meta['reference']} · prompt = {meta['prompt']}\n"
            f"duration = {len(audio)/sr:.2f}s · sample_rate = {sr} Hz · {meta['elapsed']:.1f}s wall\n")
    return (sr, audio), path, info


# ── Tab 5: Streaming ─────────────────────────────────────────────
def ui_streaming(text, version, cfg, timesteps, normalize, progress=gr.Progress()):
    if not text or not text.strip():
        raise gr.Error('Type some text first.')
    pieces = []
    sr = None
    log = []
    progress(0.0, 'Loading...')
    for i, (chunk, c_sr) in enumerate(synthesize_streaming(text, version, cfg, timesteps, normalize)):
        pieces.append(chunk)
        sr = c_sr
        log.append(f'chunk {i+1}: {len(chunk)} samples')
        progress(min((i + 1) / 100.0, 0.99), f'chunk {i+1}')
    if not pieces:
        raise gr.Error('No audio generated.')
    full = np.concatenate(pieces)
    progress(1.0, f'Done. {len(pieces)} chunks.')
    return (sr, full), '\n'.join(log)


# ── Tab 6: Batch ─────────────────────────────────────────────────
def ui_batch(text, ref_audio, version, cfg, timesteps, normalize, denoise, progress=gr.Progress()):
    if not text or not text.strip():
        raise gr.Error('Type some text first.')
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        raise gr.Error('No non-empty lines.')
    if ref_audio is not None and version == 'voxcpm-0.5b':
        raise gr.Error('Voice Clone is only supported by VoxCPM2. Either remove the reference audio or switch to VoxCPM2.')
    ref_path = None
    if ref_audio is not None:
        mat = _materialize_audio(ref_audio)
        if mat is not None:
            ref_sr, ref_data = mat
            if ref_sr is None:
                ref_sr = 48000  # Voice Clone is V2-only, default 48 kHz
            ref_path = f'/content/ref_audio/batch_ref_{int(time.time()*1000)}.wav'
            sf.write(ref_path, ref_data, ref_sr, subtype='PCM_16')
    out_dir = '/content/audio_out/batch'
    os.makedirs(out_dir, exist_ok=True)
    zip_buf = io.BytesIO()
    log = []
    with zipfile.ZipFile(zip_buf, 'w', zipfile.ZIP_DEFLATED) as zf:
        for i, line in enumerate(lines):
            try:
                if ref_path:
                    audio, sr, _ = synthesize_voice_clone(line, ref_path, cfg, timesteps, normalize, denoise, version)
                else:
                    audio, sr, _ = synthesize_tts(line, cfg, timesteps, normalize, denoise, version)
                fname = f'line_{i:03d}.wav'
                wav_path = os.path.join(out_dir, fname)
                sf.write(wav_path, audio, sr, subtype='PCM_16')
                with open(wav_path, 'rb') as f:
                    zf.writestr(fname, f.read())
                log.append(f'[{i+1:03d}/{len(lines)}] ok · {len(audio)/sr:.2f}s · {line[:40]}{"…" if len(line)>40 else ""}')
            except Exception as e:
                log.append(f'[{i+1:03d}/{len(lines)}] FAILED · {e}')
            progress((i+1)/max(len(lines), 1), f'{i+1}/{len(lines)}')
    if ref_path:
        try: os.remove(ref_path)
        except Exception: pass
    zip_path = '/content/audio_out/batch.zip'
    with open(zip_path, 'wb') as f:
        f.write(zip_buf.getvalue())
    return zip_path, '\n'.join(log)


# ── Tab 7: VRAM ──────────────────────────────────────────────────
def ui_vram_list():
    if MODEL is None:
        msg = 'No model loaded yet.'
    else:
        spec = next((v for v in VERSIONS if v[0] == MODEL_VERSION), None)
        if spec is not None:
            msg = f'Loaded: {spec[2]}'
        else:
            msg = f'Loaded: {MODEL_VERSION}'
    if not torch.cuda.is_available():
        return f'{msg}\n(CPU mode — no GPU memory to query.)'
    free, tot = torch.cuda.mem_get_info(0)
    return f'{msg}\nGPU memory: {free/1e9:.1f} / {tot/1e9:.1f} GB free'


def ui_free_all():
    free_voxcpm()
    if torch.cuda.is_available():
        free, tot = torch.cuda.mem_get_info(0)
        return f'Freed all.\nGPU memory: {free/1e9:.1f} / {tot/1e9:.1f} GB free.'
    return 'Freed all. (CPU mode — no GPU to query.)'


# ── Build the UI ────────────────────────────────────────────────
with gr.Blocks(title='VoxCPM2', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# VoxCPM2 — Tokenizer-Free Multilingual TTS\n'
                'Apache 2.0 · 27.3k★ · 2B · 30 langs · 48 kHz · OpenBMB')

    with gr.Tabs():
        # ── Tab 1: TTS ──
        with gr.Tab('TTS'):
            gr.Markdown('**Plain text-to-speech** — the model infers a fitting voice from the text itself. No reference needed. Try poetry, news, dramatic monologues.')
            with gr.Row():
                with gr.Column():
                    t1_text = gr.Textbox(label='Text', lines=4,
                                          value='VoxCPM2 is a tokenizer-free text-to-speech model that generates highly expressive speech.',
                                          info='Any text. VoxCPM2 will pick an appropriate voice based on the content.')
                    with gr.Row():
                        t1_version = gr.Dropdown([v[0] for v in VERSIONS],
                                                   value='voxcpm2', label='Model',
                                                   info='VoxCPM2 (default, 2B, 30 langs) or VoxCPM-0.5B (legacy, 0.5B, ZH/EN).')
                        t1_cfg = gr.Slider(1.0, 4.0, value=2.0, step=0.1, label='CFG value',
                                           info='How strictly the model follows the text. Lower = more expressive, higher = more adherence.')
                        t1_timesteps = gr.Slider(5, 30, value=10, step=1, label='Inference timesteps',
                                                  info='Higher = better quality, slower. 10 is a good default.')
                    t1_normalize = gr.Checkbox(label='Text normalization (numbers, abbreviations)',
                                                value=False,
                                                info='Enable for natural handling of "123" or "Dr." etc. Adds startup time on first use.')
                    t1_denoise = gr.Checkbox(label='Denoise (ZipEnhancer)',
                                              value=False,
                                              info='Acoustic noise suppression for noisy prompts. Adds ~50 MB. Not needed for TTS.')
                    t1_btn = gr.Button('🔊 Generate', variant='primary')
                with gr.Column():
                    t1_audio = gr.Audio(label='Output', type='numpy'),

                    t1_file = gr.File(label='Download .wav')
                    t1_info = gr.Textbox(label='Run info', lines=4, interactive=False)

            t1_btn.click(ui_tts,
                         [t1_text, t1_version, t1_cfg, t1_timesteps, t1_normalize, t1_denoise],
                         [t1_audio, t1_file, t1_info])

        # ── Tab 2: Voice Design ──
        with gr.Tab('Voice Design'):
            gr.Markdown('**Create a voice from a text description** — no reference audio needed. Type a description in the **Custom** field, OR pick a preset. If both are provided, the Custom field wins. (V2 only.)')
            with gr.Row():
                with gr.Column():
                    t2_text = gr.Textbox(label='Text', lines=3,
                                          value='Hello! Welcome to VoxCPM2 — a new era of voice design.',
                                          info='What the voice should say. The description will be prepended automatically in parens.')
                    t2_desc = gr.Textbox(label='Custom description (overrides preset if non-empty)', lines=2,
                                          value='A young woman, gentle and sweet voice',
                                          placeholder='e.g. "Deep male, British accent, formal"',
                                          info='Examples: "Deep male, British accent, formal", "Excited teenage girl, fast", "Warm elderly female, slow".')
                    t2_preset = gr.Dropdown(['(none)'] + VOICE_DESIGN_PRESETS,
                                             value='(none)', label='Voice description preset (used if Custom is empty)',
                                             info='Pick a preset, OR write your own in the Custom field above. The (none) option means "use the custom field".')
                    with gr.Row():
                        t2_version = gr.Dropdown([v[0] for v in VERSIONS],
                                                   value='voxcpm2', label='Model',
                                                   info='Voice Design requires VoxCPM2.')
                        t2_cfg = gr.Slider(1.0, 4.0, value=2.0, step=0.1, label='CFG value')
                        t2_timesteps = gr.Slider(5, 30, value=10, step=1, label='Inference timesteps')
                    t2_normalize = gr.Checkbox(label='Text normalization', value=False)
                    t2_denoise = gr.Checkbox(label='Denoise', value=False)
                    t2_btn = gr.Button('🎨 Generate with designed voice', variant='primary')
                with gr.Column():
                    t2_audio = gr.Audio(label='Output', type='numpy')
                    t2_file = gr.File(label='Download .wav')
                    t2_info = gr.Textbox(label='Run info', lines=4, interactive=False)

            t2_btn.click(ui_voice_design,
                         [t2_desc, t2_text, t2_preset, t2_version, t2_cfg, t2_timesteps, t2_normalize, t2_denoise],
                         [t2_audio, t2_file, t2_info])

        # ── Tab 3: Voice Clone ──
        with gr.Tab('Voice Clone'):
            gr.Markdown("**Clone the timbre from a short reference clip** (5-30 s of clear speech). The reference voice's timbre is applied to the text.")
            with gr.Row():
                with gr.Column():
                    t3_text = gr.Textbox(label='Text', lines=4,
                                          value='This is my cloned voice speaking with VoxCPM2 — natural, expressive, and faithful to the original.',
                                          info='What the cloned voice will say.')
                    t3_ref = gr.Audio(label='Reference audio (5-30s of clear speech)',
                                       type='numpy'),

                    with gr.Row():
                        t3_version = gr.Dropdown([v[0] for v in VERSIONS],
                                                   value='voxcpm2', label='Model'),

                        t3_cfg = gr.Slider(1.0, 4.0, value=2.0, step=0.1, label='CFG value')
                        t3_timesteps = gr.Slider(5, 30, value=10, step=1, label='Inference timesteps')
                    t3_normalize = gr.Checkbox(label='Text normalization', value=False)
                    t3_denoise = gr.Checkbox(label='Denoise reference audio',
                                              value=True,
                                              info='Run ZipEnhancer on the reference. Disable if your audio is already clean.')
                    t3_btn = gr.Button('🎤 Clone voice', variant='primary')
                with gr.Column():
                    t3_audio = gr.Audio(label='Output', type='numpy')
                    t3_file = gr.File(label='Download .wav')
                    t3_info = gr.Textbox(label='Run info', lines=4, interactive=False)

            t3_btn.click(ui_voice_clone,
                         [t3_text, t3_ref, t3_version, t3_cfg, t3_timesteps, t3_normalize, t3_denoise],
                         [t3_audio, t3_file, t3_info])

        # ── Tab 4: Ultimate Clone ──
        with gr.Tab('Ultimate Clone'):
            gr.Markdown('**Preserve every vocal nuance** — provide the reference audio, its transcript, and a prompt audio. VoxCPM2 continues from the prompt, faithfully reproducing the reference. Maximum similarity.')
            with gr.Row():
                with gr.Column():
                    t4_text = gr.Textbox(label='Text', lines=3,
                                          value='This is an ultimate cloning demonstration. Every vocal nuance of the original speaker is preserved.',
                                          info='What the voice will say.')
                    t4_ref = gr.Audio(label='Reference audio (5-30s)',
                                       type='numpy'),

                    t4_prompt = gr.Audio(label='Prompt audio (same as reference, or extension)',
                                          type='numpy'),

                    t4_prompt_text = gr.Textbox(label='Prompt text (transcript of the reference)',
                                                 value=''),

                    with gr.Row():
                        t4_version = gr.Dropdown([v[0] for v in VERSIONS],
                                                   value='voxcpm2', label='Model',
                                                   info='Ultimate Clone requires VoxCPM2.')
                        t4_cfg = gr.Slider(1.0, 4.0, value=2.0, step=0.1, label='CFG value')
                        t4_timesteps = gr.Slider(5, 30, value=10, step=1, label='Inference timesteps')
                    t4_normalize = gr.Checkbox(label='Text normalization', value=False)
                    t4_denoise = gr.Checkbox(label='Denoise both', value=True)
                    t4_btn = gr.Button('🎙️ Ultimate clone', variant='primary')
                with gr.Column():
                    t4_audio = gr.Audio(label='Output', type='numpy')
                    t4_file = gr.File(label='Download .wav')
                    t4_info = gr.Textbox(label='Run info', lines=4, interactive=False)

            t4_btn.click(ui_ultimate_clone,
                         [t4_text, t4_ref, t4_prompt, t4_prompt_text, t4_version,
                          t4_cfg, t4_timesteps, t4_normalize, t4_denoise],
                         [t4_audio, t4_file, t4_info])

        # ── Tab 5: Streaming ──
        with gr.Tab('Streaming'):
            gr.Markdown('**Long texts are chunked and played back as they are generated.** Each chunk is saved individually; the full concatenation is also returned.')
            with gr.Row():
                with gr.Column():
                    t5_text = gr.Textbox(label='Text (long-form works best)',
                                          lines=8,
                                          value='The quick brown fox jumps over the lazy dog. ' * 5,
                                          info='VoxCPM2 will stream audio chunks as they are produced. RTF ~0.3 on RTX 4090.')
                    with gr.Row():
                        t5_version = gr.Dropdown([v[0] for v in VERSIONS],
                                                   value='voxcpm2', label='Model')
                        t5_cfg = gr.Slider(1.0, 4.0, value=2.0, step=0.1, label='CFG value')
                        t5_timesteps = gr.Slider(5, 30, value=10, step=1, label='Inference timesteps')
                    t5_normalize = gr.Checkbox(label='Text normalization', value=False)
                    t5_btn = gr.Button('▶ Stream', variant='primary')
                with gr.Column():
                    t5_audio = gr.Audio(label='Concatenated output', type='numpy')
                    t5_log = gr.Textbox(label='Chunk log', lines=8, interactive=False),


            t5_btn.click(ui_streaming,
                         [t5_text, t5_version, t5_cfg, t5_timesteps, t5_normalize],
                         [t5_audio, t5_log])

        # ── Tab 6: Batch ──
        with gr.Tab('Batch'):
            gr.Markdown('**Generate one .wav per line** and download as a zip. Optional reference audio applies the same cloned voice to every line.')
            with gr.Row():
                with gr.Column():
                    b_text = gr.Textbox(label='Lines (one per utterance)', lines=8,
                                         value='The quick brown fox jumps over the lazy dog.\nShe sells seashells by the seashore.\nTo be, or not to be: that is the question.',
                                         info='Blank lines are skipped.')
                    b_ref = gr.Audio(label='Reference audio (optional)',
                                     type='numpy'),

                    with gr.Row():
                        b_version = gr.Dropdown([v[0] for v in VERSIONS],
                                                 value='voxcpm2', label='Model')
                        b_cfg = gr.Slider(1.0, 4.0, value=2.0, step=0.1, label='CFG value')
                        b_timesteps = gr.Slider(5, 30, value=10, step=1, label='Inference timesteps')
                    b_normalize = gr.Checkbox(label='Text normalization', value=False)
                    b_denoise = gr.Checkbox(label='Denoise', value=False)
                    b_btn = gr.Button('📦 Run batch', variant='primary')
                with gr.Column():
                    b_zip = gr.File(label='Download .zip of .wav files')
                    b_log = gr.Textbox(label='Per-line log', lines=10, interactive=False,
                                        info='One line per utterance. FAILED lines are skipped.')

            b_btn.click(ui_batch,
                        [b_text, b_ref, b_version, b_cfg, b_timesteps, b_normalize, b_denoise],
                        [b_zip, b_log])

        # ── Tab 7: VRAM ──
        with gr.Tab('VRAM'):
            gr.Markdown('VoxCPM2 needs ~8 GB VRAM; VoxCPM-0.5B needs ~5 GB. The model is auto-loaded on first use in any tab.')
            v_log = gr.Textbox(label='Loaded model', lines=3, interactive=False,
                                info='Use the Free button to release the model and clear VRAM.')
            v_btn = gr.Button('🧹 Free loaded model', variant='secondary')
            v_btn.click(ui_free_all, outputs=v_log)
            demo.load(ui_vram_list, outputs=v_log)

        # ── Tab 8: Help ──
        with gr.Tab('Help'):
            gr.Markdown(
                """## 30 supported languages

VoxCPM2: Arabic, Burmese, Chinese, Danish, Dutch, English, Finnish, French, German, Greek, Hebrew, Hindi, Indonesian, Italian, Japanese, Khmer, Korean, Lao, Malay, Norwegian, Polish, Portuguese, Russian, Spanish, Swahili, Swedish, Tagalog, Thai, Turkish, Vietnamese.

Chinese dialects: 四川话, 粤语, 吴语, 东北话, 河南话, 陕西话, 山东话, 天津话, 闽南话.

VoxCPM-0.5B is **bilingual only** (Chinese + English).

## Four generation modes

| Mode | Reference audio? | Best for | VoxCPM-0.5B? |
| --- | --- | --- | --- |
| **TTS** | No | General narration, content-voice matching | Yes |
| **Voice Design** | No | Creating a new voice from a description | No (V2 only) |
| **Voice Clone** | Yes (5-30 s) | Cloning a known voice | No (V2 only) |
| **Ultimate Clone** | Yes + transcript | Maximum-fidelity continuation | No (V2 only) |

## How Voice Design works

The model treats text in parentheses as a **voice description**, not literal content. The rest of the text is what the voice will say. Examples:

- `(A young woman, gentle and sweet voice)Hello, welcome!`
- `(An elderly British man, deep and slow)Good evening.`
- `(Excited child)Look at the puppy!`

The more specific the description (gender, age, accent, emotion, pace, register), the more reliably the model hits the target.

## Inference tuning

- **CFG value** (1.0 - 4.0, default 2.0) — LM guidance. Lower = more expressive/prosodic. Higher = more text-adherent but may sound strained.
- **Inference timesteps** (5 - 30, default 10) — diffusion steps. Higher = better quality, slower.
- **Text normalization** — On for natural handling of numbers ("123" → "one hundred twenty-three"). Off for direct control (e.g. "{HH AH0 L OW1}" phoneme input).

## Performance vs other open TTS

VoxCPM2 (2B) is the **highest-SIM** open model on Seed-TTS-eval and MiniMax-MLS-test (across 20+ languages). Comparable to MegaTTS3, Seed-TTS, and other closed-source systems.

## License

[Apache-2.0](https://github.com/OpenBMB/VoxCPM/blob/main/LICENSE) — free for commercial use, including the cloned voices you create with this notebook.

## Citation

```bibtex
@article{voxcpm2025,
  title  = {VoxCPM: Tokenizer-Free TTS for Context-Aware Speech Generation and True-to-Life Voice Cloning},
  author = {{VoxCPM Team}},
  journal= {arXiv preprint arXiv:2509.24650},
  year   = {2025}
}
```
"""
            )

# ── Welcome message on first load ──
def _welcome():
    return ('🎙️ VoxCPM2 ready. 2B params, 30 languages, 48 kHz. '
            'Start with the **TTS** tab (no ref needed) or **Voice Design** (description in parens). '
            'For voice cloning use **Voice Clone** (V2 only).')

demo.load(_welcome, outputs=[])

demo.queue(max_size=8, concurrency_limit=2).launch(
    share=True, show_error=True, server_name='0.0.0.0', server_port=7860,
)


In [ ]:
# @title Step 5 — Keep Alive
# @markdown Prevents Colab from disconnecting during long sessions. Run anytime after launching the UI.

import threading, time
def _keep():
    while True:
        time.sleep(60)
        try:
            import requests
            requests.get('https://www.google.com', timeout=5)
        except Exception:
            pass
threading.Thread(target=_keep, daemon=True).start()
print('[Keep-Alive] Running. Stop cell to disable.')


In [ ]:
# @title Step 6 — Quick Test (one-off generation, no UI)
# @markdown Run a single VoxCPM generation from the notebook. Useful for scripting or quick checks.

MODE = "voice_design"  # @param ["tts", "voice_design", "voice_clone", "ultimate_clone"]
VERSION = "voxcpm2"  # @param ["voxcpm2", "voxcpm-0.5b"]
TEXT = "The warm morning sun cast golden rays across the quiet valley, awakening the world with a gentle embrace."  # @param {type:"string"}
DESCRIPTION = "(A warm, professional female narrator, clear and engaging, audiobook quality)"  # @param {type:"string"}
REF_AUDIO = ""  # @param {type:"string"}
PROMPT_AUDIO = ""  # @param {type:"string"}
PROMPT_TEXT = ""  # @param {type:"string"}
CFG_VALUE = 2.0  # @param {type:"slider", min:1.0, max:4.0, step:0.1}
TIMESTEPS = 20  # @param {type:"slider", min:5, max:30, step:1}
NORMALIZE = False  # @param {type:"boolean"}
DENOISE = False  # @param {type:"boolean"}

import os, time
if not TEXT or not TEXT.strip():
    raise SystemExit('TEXT is required.')

print(f'[Test] mode={MODE} version={VERSION} cfg={CFG_VALUE:.2f} timesteps={TIMESTEPS}')
print(f'[Test] text = {TEXT!r}')

t0 = time.time()
if MODE == 'tts':
    audio, sr, meta = synthesize_tts(TEXT, CFG_VALUE, TIMESTEPS, NORMALIZE, DENOISE, VERSION)
elif MODE == 'voice_design':
    if VERSION == 'voxcpm-0.5b':
        raise SystemExit('Voice Design is only supported by VoxCPM2.')
    audio, sr, meta = synthesize_voice_design(DESCRIPTION, TEXT, CFG_VALUE, TIMESTEPS, NORMALIZE, DENOISE, VERSION)
elif MODE == 'voice_clone':
    if not REF_AUDIO or not os.path.exists(REF_AUDIO):
        raise SystemExit('REF_AUDIO is required for voice_clone mode (path to a wav/mp3 file).')
    if VERSION == 'voxcpm-0.5b':
        raise SystemExit('Voice Clone is only supported by VoxCPM2.')
    audio, sr, meta = synthesize_voice_clone(TEXT, REF_AUDIO, CFG_VALUE, TIMESTEPS, NORMALIZE, DENOISE, VERSION)
elif MODE == 'ultimate_clone':
    if not REF_AUDIO or not PROMPT_AUDIO or not PROMPT_TEXT:
        raise SystemExit('REF_AUDIO, PROMPT_AUDIO, and PROMPT_TEXT are all required for ultimate_clone.')
    if VERSION == 'voxcpm-0.5b':
        raise SystemExit('Ultimate Clone is only supported by VoxCPM2.')
    audio, sr, meta = synthesize_ultimate_clone(TEXT, REF_AUDIO, PROMPT_AUDIO, PROMPT_TEXT,
                                                 CFG_VALUE, TIMESTEPS, NORMALIZE, DENOISE, VERSION)
else:
    raise SystemExit(f'Unknown mode: {MODE}')

elapsed = time.time() - t0
path = save_wav(audio, sr, f'quicktest_{int(time.time())}.wav')
print(f'\n[Done] {path}')
print(f'[Done] {len(audio)} samples · {len(audio)/sr:.2f}s @ {sr} Hz · generated in {elapsed:.1f}s')

from IPython.display import Audio, display
display(Audio(path, autoplay=False))


In [ ]:
# @title Step 7 — Batch Synthesis
# @markdown Generate audio for a list of texts. Each non-empty line in `BATCH` becomes one output file.

BATCH_MODE = "voice_design"  # @param ["tts", "voice_design", "voice_clone"]
BATCH_VERSION = "voxcpm2"  # @param ["voxcpm2", "voxcpm-0.5b"]
BATCH_REF_AUDIO = ""  # @param {type:"string"}
BATCH_DESCRIPTION = "(A warm, professional female narrator, clear and engaging, audiobook quality)"  # @param {type:"string"}
BATCH_CFG = 2.0  # @param {type:"slider", min:1.0, max:4.0, step:0.1}
BATCH_TIMESTEPS = 10  # @param {type:"slider", min:5, max:30, step:1}
BATCH_NORMALIZE = False  # @param {type:"boolean"}
BATCH_DENOISE = False  # @param {type:"boolean"}
BATCH_OUT_DIR = "/content/audio_out/batch"  # @param {type:"string"}

BATCH = """\
The quick brown fox jumps over the lazy dog.
She sells seashells by the seashore.
To be, or not to be: that is the question.
It was the best of times, it was the worst of times.
All happy families are alike; each unhappy family is unhappy in its own way.
"""

import os, time
lines = [l.strip() for l in BATCH.strip().splitlines() if l.strip()]
if not lines:
    raise SystemExit('BATCH is empty.')

if BATCH_MODE == 'voice_clone':
    if not BATCH_REF_AUDIO or not os.path.exists(BATCH_REF_AUDIO):
        raise SystemExit('BATCH_REF_AUDIO is required for voice_clone mode.')
    if BATCH_VERSION == 'voxcpm-0.5b':
        raise SystemExit('Voice Clone is only supported by VoxCPM2.')

os.makedirs(BATCH_OUT_DIR, exist_ok=True)
print(f'[Batch] {len(lines)} lines · mode={BATCH_MODE} version={BATCH_VERSION} cfg={BATCH_CFG:.2f} timesteps={BATCH_TIMESTEPS}')

t_start = time.time()
for i, line in enumerate(lines):
    label = f"[{i+1:03d}/{len(lines)}]"
    snippet = line[:60] + ('…' if len(line) > 60 else '')
    print(f'{label} {snippet}', flush=True)
    t0 = time.time()
    try:
        if BATCH_MODE == 'tts':
            audio, sr, _ = synthesize_tts(line, BATCH_CFG, BATCH_TIMESTEPS, BATCH_NORMALIZE, BATCH_DENOISE, BATCH_VERSION)
        elif BATCH_MODE == 'voice_design':
            audio, sr, _ = synthesize_voice_design(BATCH_DESCRIPTION, line, BATCH_CFG, BATCH_TIMESTEPS, BATCH_NORMALIZE, BATCH_DENOISE, BATCH_VERSION)
        else:
            audio, sr, _ = synthesize_voice_clone(line, BATCH_REF_AUDIO, BATCH_CFG, BATCH_TIMESTEPS, BATCH_NORMALIZE, BATCH_DENOISE, BATCH_VERSION)
        out_path = os.path.join(BATCH_OUT_DIR, f'{i:03d}.wav')
        sf.write(out_path, audio, sr, subtype='PCM_16')
        print(f'  ✓ {len(audio)/sr:.2f}s · {time.time()-t0:.1f}s · {out_path}', flush=True)
    except Exception as e:
        print(f'  ✗ EXCEPTION: {e}', flush=True)
        continue

elapsed = time.time() - t_start
print(f'\n[Done] {len(lines)} files in {BATCH_OUT_DIR} · total wall time {elapsed:.1f}s')
